# Temamodellering med Topeax

Vi finner temaer i subsettet med Topeax-algoritmen fra [turftopic](https://github.com/x-tabdeveloping/turftopic), og bruker den samme Borealis-modellen som i `borealis_embeddings.ipynb` til å lage dokumentembeddingene.

Topeax flater ut embeddingene til 2D med t-SNE og finner topper i tettheten av dokumenter.  
Hver topp blir et tema, så antall temaer blir funnet automatisk.  

Nøkkelordene til et tema vektes både etter hvor typiske de er for temaets dokumenter (NPMI) og hvor like embeddingene deres er temaets embedding.

## Les inn subsettet

In [ ]:
from pathlib import Path

import pandas as pd

SUBSET_DIR = Path("data/maalfrid/subset")
paths = sorted(SUBSET_DIR.glob("*/*.txt"))
texts = pd.Series(
    [path.read_text(encoding="utf-8") for path in paths],
    index=pd.Index([f"{path.parent.name}/{path.stem}" for path in paths]),
)
domains = texts.index.str.split("/").str[0]
len(texts)

## Lag embeddinger og finn temaer med Topeax


In [ ]:
import gc

import torch
from sentence_transformers import SentenceTransformer
from turftopic import Topeax

model = SentenceTransformer("NbAiLab/borealis-embed-212m")
embeddings = model.encode(texts.tolist(), batch_size=4, show_progress_bar=True)


topic_model = Topeax(encoder=model, random_state=42)
doc_topic_matrix = topic_model.fit_transform(texts.tolist(), embeddings=embeddings)

del model
gc.collect()
torch.cuda.empty_cache()

## Temaer per domene

Hvert dokument tilordnes temaet det har høyest sannsynlighet for.  
Tabellen viser hvor mange dokumenter fra hvert domene som havner i hvert tema.

In [ ]:
topic_labels = [
    f"{topic_id}: " + ", ".join(word for word, _ in words)
    for topic_id, words in topic_model.get_topics(top_k=3)
]
doc_topics = pd.Series(
    [topic_labels[label] for label in topic_model.labels_], index=texts.index
)
pd.crosstab(doc_topics, domains, rownames=["tema"], colnames=["domene"])

## Visualisering av topeax
Tettheten av punkter lager "fjelltopper". Hver fjelltopp blir et tema.

In [ ]:
topic_model.plot_density(show_keywords=False)

Prikkene er dokumentene, farget etter temaet de blir kategorisert i.  
Hold over en prikk for å se kildedomenet.

In [ ]:
topic_model.plot_components(
    show_points=True, show_keywords=False, hover_text=domains.tolist()
)

## Temaer og domener i embedding-dimensjonene
(Samme dimensjoner som i borealis_embeddings-notebooken)

Fargen viser temaet dokumentet er tilordnet, og formen viser kildedomenet.  


In [ ]:
from visualize import plot_topics_by_domain, project_2d

projection = project_2d(embeddings, texts.index)
plot_topics_by_domain(projection, doc_topics, texts, "Topeax topics by domain")